# Train the expression-rate and size-factor branches

This notebook exposes the Python training APIs, model architectures,
target definitions and checkpoint contents. It does not launch opaque
shell commands. Full HEST training is deliberately guarded because it is
a long GPU job and requires the processed arrays described in notebook 01.


In [ ]:
from pathlib import Path
import copy
import sys

import pandas as pd
import torch
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the HistoOmniST repository.")


ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

from histoomnist.models.sf_model import SizeFactorRegressor
from histoomnist.train.train_expression import build_rate_model, train as train_rate
from histoomnist.train.train_sf import build_loss, train as train_sf
from histoomnist.utils.config import load_config


## 1. Load the frozen training configurations


In [ ]:
rate_config_path = ROOT / "configs/hest1k_human_visium_expression_highconf_symbol95.yaml"
sf_config_path = ROOT / "configs/hest1k_human_visium_sf_context_distribution_light.yaml"
rate_config = load_config(rate_config_path)
sf_config = load_config(sf_config_path)

configuration_summary = pd.DataFrame(
    [
        {
            "branch": "expression rate",
            "target": rate_config["model"]["output"],
            "model": rate_config["model"]["name"],
            "epochs": rate_config["training"]["epochs"],
            "batch_size": rate_config["training"]["batch_size"],
            "learning_rate": rate_config["training"]["lr"],
        },
        {
            "branch": "size factor",
            "target": sf_config["model"]["output"],
            "model": sf_config["model"]["architecture"],
            "epochs": sf_config["training"]["epochs"],
            "batch_size": sf_config["training"]["batch_size"],
            "learning_rate": sf_config["training"]["lr"],
        },
    ]
)
display(configuration_summary)


## 2. Confirm slides and genes used by training

The rate branch uses 16,942 canonical symbols observed in at least 95%
of the reporting slides. Both branches use fixed slide-level partitions.


In [ ]:
rate_manifest = pd.read_csv(ROOT / rate_config["data"]["manifest"])
sf_manifest = pd.read_csv(ROOT / sf_config["data"]["manifest"])
gene_path = (ROOT / "data/HEST-1k/manifests" / rate_config["data"]["gene_names_path"]).resolve()
genes = gene_path.read_text(encoding="utf-8").splitlines()

print(f"Expression target genes: {len(genes):,}")
print("First ten genes:", genes[:10])
display(rate_manifest["split"].value_counts().rename_axis("split").to_frame("rate slides"))
display(sf_manifest["split"].value_counts().rename_axis("split").to_frame("SF slides"))


## 3. Instantiate the two model architectures


In [ ]:
INPUT_DIM = 1161
rate_model = build_rate_model(rate_config, input_dim=INPUT_DIM, output_dim=len(genes))
sf_model = SizeFactorRegressor(
    input_dim=INPUT_DIM,
    hidden_dims=list(sf_config["model"].get("hidden_dims") or []),
    dropout=float(sf_config["model"]["dropout"]),
    architecture=str(sf_config["model"]["architecture"]),
    width=int(sf_config["model"]["width"]),
    depth=int(sf_config["model"]["depth"]),
)
sf_loss = build_loss(sf_config)

def trainable_parameters(model: torch.nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

display(
    pd.DataFrame(
        [
            {"branch": "expression rate", "parameters": trainable_parameters(rate_model), "output_dim": len(genes)},
            {"branch": "size factor", "parameters": trainable_parameters(sf_model), "output_dim": 1},
        ]
    )
)
print("SF loss:", sf_loss)


## 4. Run full training through the Python APIs

Set `RUN_FULL_TRAINING = True` only after notebook 01 confirms that all
manifest paths resolve. The functions below create datasets, fit feature
standardization on training slides only, use validation-selected early
stopping and save complete checkpoint metadata.


In [ ]:
RUN_FULL_TRAINING = False

if RUN_FULL_TRAINING:
    rate_runtime = copy.deepcopy(rate_config)
    sf_runtime = copy.deepcopy(sf_config)
    rate_runtime["device"] = "cuda" if torch.cuda.is_available() else "cpu"
    sf_runtime["device"] = rate_runtime["device"]

    current_directory = Path.cwd()
    try:
        import os
        os.chdir(ROOT)
        rate_checkpoint = train_rate(rate_runtime)
        sf_checkpoint = train_sf(sf_runtime)
    finally:
        os.chdir(current_directory)

    print("Rate checkpoint:", rate_checkpoint)
    print("SF checkpoint:", sf_checkpoint)
else:
    print("Training not started. Set RUN_FULL_TRAINING=True after preparing HEST arrays.")


## 5. Checkpoint contract

The expression checkpoint stores 16,942 gene names, the model state,
training-only feature mean/std and the complete configuration. The SF
checkpoint stores the same feature-standardization contract and predicts
raw `log(SF)`. At inference, predicted SF values are exponentiated and
renormalized to mean one separately for each slide before reconstruction.
